# Bài 11 - Giao thức Ngữ cảnh Mô hình (MCP)

**Giao thức Ngữ cảnh Mô hình (MCP)** là một tiêu chuẩn mở cho phép các tác nhân phát hiện và sử dụng công cụ, tài nguyên và nguồn dữ liệu một cách động khi chạy. Thay vì mã hóa cứng các công cụ vào trong tác nhân, MCP cho phép các tác nhân kết nối với các máy chủ bên ngoài cung cấp các khả năng theo yêu cầu.

Trong bài học này, bạn sẽ tìm hiểu:
- MCP là gì và tại sao nó quan trọng đối với các hệ thống tác nhân
- Cách kiến trúc máy khách-máy chủ MCP hoạt động
- Cách xây dựng các tác nhân sử dụng khám phá công cụ theo kiểu MCP


## Cài đặt

**Yêu cầu trước:**
- Dự án Microsoft Foundry với một mô hình đã triển khai
- Chạy `az login` để xác thực


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import dotenv
from typing import Annotated
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Giao thức Ngữ cảnh Mô hình (MCP) là gì?

MCP định nghĩa một cách chuẩn để các tác nhân AI khám phá và tương tác với các công cụ và nguồn dữ liệu bên ngoài:

- **MCP Server**: Cung cấp công cụ, tài nguyên và lời nhắc qua một giao thức chuẩn
- **MCP Client**: Thời gian chạy của tác nhân kết nối với các máy chủ và khám phá các khả năng có sẵn
- **Khám phá động**: Các tác nhân không cần công cụ mã cứng — họ khám phá những gì có sẵn khi chạy

Điều này rất mạnh mẽ để xây dựng các hệ thống tác nhân có thể mở rộng nơi các khả năng mới có thể được thêm vào mà không cần sửa đổi mã của tác nhân.


## Cách MCP Hoạt Động

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Agent (khách hàng MCP) kết nối đến máy chủ MCP
2. Máy chủ trả về danh sách các công cụ có sẵn và sơ đồ của chúng
3. Agent có thể gọi bất kỳ công cụ nào được phát hiện trong quá trình suy luận của nó
4. Kết quả được trả về qua cùng một giao thức


## Mô phỏng việc Phát hiện Công cụ MCP

Vì một máy chủ MCP thực sự yêu cầu một tiến trình máy chủ đang chạy, chúng tôi sẽ trình bày mẫu này bằng cách sử dụng các hàm `@tool` mô phỏng những gì một dịch vụ chỗ ở kết nối MCP sẽ cung cấp.

Trong môi trường sản xuất, các công cụ này sẽ được phát hiện động từ một máy chủ MCP thay vì được định nghĩa cục bộ.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Xây dựng một Đại lý với Công cụ kiểu MCP


In [ ]:
agent = client.as_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP trong Môi trường Sản xuất

Trong môi trường sản xuất, MCP cho phép các mô hình mạnh mẽ:

- **Khám phá công cụ động**: Các tác vụ kết nối với máy chủ MCP và khám phá công cụ trong thời gian chạy
- **Kiến trúc tách rời**: Nhà cung cấp công cụ có thể cập nhật độc lập với tác vụ
- **Chia sẻ giữa các tổ chức**: Các nhóm có thể cung cấp khả năng thông qua máy chủ MCP mà bất kỳ tác vụ nào cũng có thể sử dụng
- **Hỗ trợ Microsoft Agent Framework**: MAF có hỗ trợ khách hàng MCP tích hợp sẵn qua `mcp`

Để sử dụng máy chủ MCP thực tế với MAF, bạn sẽ kết nối qua `hosted_mcp_tool()` hoặc tích hợp khách hàng MCP.

**Tìm hiểu thêm:**
- [Thông số kỹ thuật MCP](https://modelcontextprotocol.io/)
- [Hỗ trợ MCP của Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Tóm tắt

Trong bài học này, bạn đã học:
- **MCP** là một chuẩn mở cho khám phá công cụ động giữa các tác nhân và nhà cung cấp công cụ
- **Kiến trúc máy khách-máy chủ** cho phép các tác nhân khám phá khả năng trong thời gian chạy
- MCP cho phép **hệ thống tác nhân mở rộng, tách rời** nơi công cụ có thể được thêm mà không cần thay đổi mã
- Microsoft Agent Framework cung cấp **hỗ trợ MCP tích hợp sẵn** cho sử dụng trong sản xuất


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
